In [1]:
# Save mortality by year and for each ensemble

In [2]:
import os
import xarray as xr
import numpy as np
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [4]:
# === Load data ===
bmr_file = "GBD_BMR_Country_Mask_COPD_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)

In [5]:
# === Calculate beta for RR GBD 2021 ===

# Relative risk for 10ppb increase in OSDMA8, GBD 2021
RR_10ppb = 1.074  # [95% CI 1.014 – 1.137]

# equation is: RR = e^(beta*(x-TMREL)) where RR_10ppb = e^(10beta)
beta = np.log(RR_10ppb)/10

# TMREL from GBD 2021
TMREL = 32.4  # [95% CI 29.1 – 35.7]

In [6]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        o3_file = f"OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        o3_path = os.path.join(O3_DIR, o3_file)
        o3 = xr.open_dataarray(o3_path)

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        o3 = o3.reindex_like(BMR, method="nearest", tolerance=1e-9, fill_value=0)
        population = population.reindex_like(BMR, method="nearest", tolerance=1e-9)

        # Flag if any nans present (i.e. reindex was out of tolerance distance)
        assert not population.isnull().any()

        M = []

        for year in o3["year"].values:
            AF = att_frac(o3.sel(year=year), TMREL, beta)
            POP = population.sel(year=year)
            mortality_year = mortality(AF, BMR, POP)
            M.append(mortality_year)

            global_mortality = mortality_year.sum(dim=("lat", "lon"))
            mean_mortality = global_mortality.sel(quantile="mean").round().values
            print(f"Mean mortality rate for {year} is {mean_mortality}")

        M_cleaned = [da.drop_vars("year", errors="ignore") for da in M]
        mortality_timeseries = xr.concat(M_cleaned,
                                         dim=(xr.DataArray(o3["year"].values,
                                                           dims="year",
                                                           name="year")))

        out_file = f"Mortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving mortality timeseries to {SAVE_DIR}")
        description = ("Total COPD mortality due to surface ozone - scripts "
                       "by A.F. Wells (2025)")
        mortality_timeseries.attrs["description"] = description
        mortality_timeseries.attrs["ensemble_number"] = ens_num
        mortality_timeseries.attrs["scenario"] = scenario
        mortality_timeseries.to_netcdf(out_path)

Processing ARISE, Ensemble 01
Mean mortality rate for 2035 is 489225.0
Mean mortality rate for 2036 is 503945.0
Mean mortality rate for 2037 is 530608.0
Mean mortality rate for 2038 is 526377.0
Mean mortality rate for 2039 is 513775.0
Mean mortality rate for 2040 is 521261.0
Mean mortality rate for 2041 is 503670.0
Mean mortality rate for 2042 is 516664.0
Mean mortality rate for 2043 is 510823.0
Mean mortality rate for 2044 is 521807.0
Mean mortality rate for 2045 is 522870.0
Mean mortality rate for 2046 is 523859.0
Mean mortality rate for 2047 is 514291.0
Mean mortality rate for 2048 is 510105.0
Mean mortality rate for 2049 is 506335.0
Mean mortality rate for 2050 is 492977.0
Mean mortality rate for 2051 is 507096.0
Mean mortality rate for 2052 is 498889.0
Mean mortality rate for 2053 is 485149.0
Mean mortality rate for 2054 is 495090.0
Mean mortality rate for 2055 is 489370.0
Mean mortality rate for 2056 is 475105.0
Mean mortality rate for 2057 is 494842.0
Mean mortality rate for 205